# Band-adapted ordinal-delay validation for HCFIM EEG

This diagnostic notebook tests whether the low ordinal entropy observed at 500 Hz with $m=4,\tau=1$ is primarily a consequence of sampling redundancy and bandpass smoothness. Delay feasibility is evaluated without diagnostic labels; labels enter only in explicitly post hoc coordinate tests and participant-level predictive sensitivity analyses.

## tl;dr

- Temporal-alpha median normalized entropy increased from **0.345** at $\tau=1$ to **0.836** at the prespecified quarter-cycle $\tau=12$; monotonic-pattern share decreased from **0.916** to **0.069**.
- The label-free median first non-positive autocorrelation lag was **13.0 samples** in alpha and **117.0 samples** in broadband. Alpha therefore supports a delay near the prespecified band-adapted scale, whereas broadband has no unique center-frequency delay.
- After alpha filtering, 50 Hz with $\tau=1$ closely reproduced the equivalent 20 ms geometry at 500 Hz with $\tau=10$ (subject-level H Spearman $\rho$ = **0.979**), directly demonstrating sampling-scale dependence.
- In the focused three-coordinate temporal models, fixed-delay mean repeated ROC AUC changed from **0.737** at $\tau=1$ to **0.776** at $\tau=12$ for AD, and from **0.677** to **0.661** for FTD. Inner-fold selection yielded **0.798** and **0.693**, respectively.
- Fold-local preferences were not universal (control vs alzheimer: τ=24 in 46%; control vs frontotemporal dementia: τ=3 in 66%). FDR-significant coordinate counts across 24 delay-coordinate tests were control vs alzheimer: 22/24; control vs frontotemporal dementia: 3/24.
- The defensible manuscript correction is to describe $m=4,\tau=1$ as a fine-scale, sampling-dependent geometry and to present band-adapted delay results as a sensitivity analysis. Neither delay should be promoted as universally optimal without external validation.


## Context & Methods

### Key assumptions

- The audit uses the first 60 valid 10 s epochs per participant, balancing analyzed duration across all 88 participants.
- The temporal waveform is constructed exactly as in the manuscript: T7/T8/P7/P8 are averaged in the time domain before fourth-order zero-phase Butterworth filtering.
- Ordinal Shannon probabilities use $m=4$ and lexicographic state ordering. The delay grid is $\tau\in\{1,2,3,6,10,12,18,24\}$ samples.
- For delta, theta, alpha, and beta, the fixed label-free band delay is the first zero of an ideal center-frequency sinusoid, $\tau_b=\mathrm{round}[f_s/(4f_{c,b})]$. Broadband has no unique physiological center frequency and is therefore evaluated only across the explicit grid and empirical autocorrelation diagnostics.
- The empirical first non-positive autocorrelation lag is descriptive and never chosen using diagnostic separation.
- The 50 Hz alpha control is performed after alpha filtering. At 50 Hz, $\tau=1$ corresponds to 20 ms, matching $\tau=10$ at 500 Hz and approximating the prespecified alpha delay $\tau_\alpha=12$ (24 ms).
- Coordinate tests and temporal H/C/FIM classification use manuscript-highlighted geometries and are post hoc sensitivity analyses, not independent confirmation or replacements for the primary full-feature models.
- Adaptive predictive selection of $\tau$ is restricted to inner folds of each outer training partition.

In [ ]:
from __future__ import annotations

import itertools
import json
import math
import os
import platform
import sys
import time
import warnings
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from IPython.display import Markdown, display
from joblib import Parallel, delayed
from scipy import fft, signal, stats
from scipy.io import loadmat
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED = 17536
N_FIXED_EPOCHS = 60
N_REPEATS = 10
OUTER_FOLDS = 5
INNER_FOLDS = 3
N_TREES = 100
MODEL_DEPTH = 3
PARALLEL_JOBS = min(4, os.cpu_count() or 1)
M = 4
TAU_GRID = (1, 2, 3, 6, 10, 12, 18, 24)
ALPHA_DECIMATED_FS = 50.0

def resolve_project_root() -> Path:
    """Locate the repository without relying on a machine-specific path."""
    env_root = os.environ.get("HCFIM_PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
        raise FileNotFoundError(
            "HCFIM_PROJECT_ROOT does not point to the HCFIM-eeg-research repository."
        )

    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository. Run the notebook from within the repository "
        "or set HCFIM_PROJECT_ROOT."
    )

PROJECT_ROOT = resolve_project_root()
def resolve_dataset_root(project_root: Path) -> Path:
    """Locate ds004504 derivatives or use HCFIM_DATASET_ROOT."""
    candidates = []
    env_root = os.environ.get("HCFIM_DATASET_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([
        project_root / "data" / "ds004504",
        project_root / "data" / "data-08-00095",
        project_root / "data",
    ])
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "participants.tsv").is_file() and (candidate / "derivatives").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate ds004504. Set HCFIM_DATASET_ROOT to the dataset directory "
        "containing participants.tsv and derivatives/."
    )

DATASET_ROOT = resolve_dataset_root(PROJECT_ROOT)
RESULTS = PROJECT_ROOT / "results"
SRC = PROJECT_ROOT / "src"
OUT = Path(os.environ.get("HCFIM_NOTEBOOK09_OUT", str(RESULTS / "band_adapted_ordinal_delay_validation")))
FIGURES = OUT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
import itm_geometry as geom

sns.set_theme(style="whitegrid", context="notebook")
COLORS = {1: "#315B7D", 2: "#6F92A8", 3: "#8DAA9D", 6: "#D18B32", 10: "#B76E79", 12: "#8A5A9E", 18: "#6F7D3C", 24: "#9C755F", "adaptive": "#444444"}

print(f"Python {platform.python_version()} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}")
print(f"Project: {PROJECT_ROOT}\nDataset: {DATASET_ROOT}\nOutput: {OUT}")

## Data

### 1. Load and validate participant inputs

In [ ]:
participants = pd.read_csv(RESULTS / "participants_index.csv")
assert len(participants) == 88
assert participants["participant_id"].is_unique
assert participants["set_exists"].all()
assert participants["diagnosis"].value_counts().to_dict() == {
    "alzheimer": 36,
    "control": 29,
    "frontotemporal_dementia": 23,
}
display(participants.groupby("diagnosis").size().rename("participants").to_frame())

## Results

### 2. Recompute temporal ordinal distributions across delays

The code below records every epoch-level H/C/FIM coordinate, ordinal-state occupancy, monotonic-pattern share, and empirical first autocorrelation zero. Filtering is performed once per epoch and band before evaluating the requested delays.

In [ ]:
CHANNEL_RENAME = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}
TEMPORAL_CHANNELS = ("T7", "T8", "P7", "P8")
BANDS = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "broadband": (0.5, 40.0),
}

def band_center(band):
    low, high = band
    return (low + high) / 2.0

def adapted_tau(sfreq, band):
    return int(round(sfreq / (4.0 * band_center(band))))

BAND_TAU_500 = {name: adapted_tau(500.0, band) for name, band in BANDS.items() if name != "broadband"}
assert BAND_TAU_500 == {"delta": 56, "theta": 21, "alpha": 12, "beta": 6}
display(pd.DataFrame([
    {"band": name, "range_hz": f"{low}-{high}", "center_hz": band_center((low, high)), "adapted_tau_samples": BAND_TAU_500.get(name, np.nan), "adapted_lag_ms": 2 * BAND_TAU_500.get(name, np.nan)}
    for name, (low, high) in BANDS.items()
]))

def load_set_matrix(set_path: Path):
    payload = loadmat(set_path, simplify_cells=True)
    data = np.asarray(payload["data"], dtype=float)
    sfreq = float(payload["srate"])
    channel_locations = payload["chanlocs"]
    if isinstance(channel_locations, dict):
        channel_locations = [channel_locations]
    labels = [CHANNEL_RENAME.get(str(item["labels"]), str(item["labels"])) for item in channel_locations]
    return data, sfreq, labels

def epoch_slices(n_samples, sfreq, n_epochs=N_FIXED_EPOCHS):
    window = int(round(10.0 * sfreq))
    step = int(round(window * 0.5))
    for epoch_id, start in enumerate(range(0, n_samples - window + 1, step)):
        if epoch_id >= n_epochs:
            break
        yield epoch_id, start, start + window

@lru_cache(maxsize=None)
def band_sos(sfreq, low, high):
    nyquist = sfreq / 2.0
    return signal.butter(4, [max(low, 0.01) / nyquist, min(high, nyquist - 0.5) / nyquist], btype="bandpass", output="sos")

def filter_window(values, sfreq, band):
    values = signal.detrend(np.asarray(values, float), type="constant")
    return signal.sosfiltfilt(band_sos(float(sfreq), *band), values)

def first_nonpositive_acf(values, max_lag=250):
    centered = np.asarray(values, float) - np.mean(values)
    nfft = fft.next_fast_len(2 * centered.size - 1)
    spectrum = fft.rfft(centered, nfft)
    acf = fft.irfft(spectrum * np.conjugate(spectrum), nfft)[: max_lag + 1]
    if acf[0] <= 0:
        return np.nan
    acf /= acf[0]
    candidates = np.flatnonzero(acf[1:] <= 0)
    return float(candidates[0] + 1) if candidates.size else np.nan

def ordinal_coordinates(values, tau):
    probabilities = geom.ordinal_probabilities(np.asarray(values, float), m=M, tau=int(tau))
    spec = geom.EntropySpec(
        entropy_id=f"ordinal_shannon_m{M}_t{int(tau)}", label=f"m{M} tau{int(tau)}", family="ordinal",
        probability_domain="Bandt-Pompe ordinal patterns", n_states=math.factorial(M),
        entropy_kind="shannon", m=M, tau=int(tau),
    )
    point = geom.hcfim_point(probabilities, spec, distance_code=geom.DISTANCES["euclidean"]["code"])
    return {
        "H": point["H"], "C": point["C"], "FIM": point["FIM"],
        "occupancy_fraction": float(np.mean(probabilities > 0)),
        "monotonic_pattern_share": float(probabilities[0] + probabilities[-1]),
        "maximum_pattern_share": float(np.max(probabilities)),
    }

def audit_subject(row):
    data, sfreq, labels = load_set_matrix(Path(row.set_path))
    if sfreq != 500.0:
        raise ValueError(f"Unexpected sampling rate {sfreq} for {row.participant_id}")
    channel_index = {label: index for index, label in enumerate(labels)}
    indices = [channel_index[channel] for channel in TEMPORAL_CHANNELS]
    temporal = np.nanmean(data[indices, :], axis=0)
    delay_records, resampling_records = [], []
    epoch_count = 0
    for epoch_id, start, end in epoch_slices(temporal.size, sfreq):
        epoch_count += 1
        raw_epoch = temporal[start:end]
        for band_name, band in BANDS.items():
            filtered = filter_window(raw_epoch, sfreq, band)
            acf_tau = first_nonpositive_acf(filtered)
            if band_name in {"alpha", "broadband"}:
                taus = TAU_GRID
            else:
                taus = tuple(sorted({1, BAND_TAU_500[band_name]}))
            for tau in taus:
                record = {
                    "participant_id": row.participant_id, "diagnosis": row.diagnosis,
                    "epoch_id": epoch_id, "band": band_name, "tau": int(tau),
                    "lag_ms": 1000.0 * tau / sfreq, "acf_first_nonpositive_tau": acf_tau,
                    "is_tau1": tau == 1,
                    "is_band_adapted": band_name != "broadband" and tau == BAND_TAU_500[band_name],
                }
                record.update(ordinal_coordinates(filtered, tau))
                delay_records.append(record)
            if band_name == "alpha":
                decimated = filtered[:: int(round(sfreq / ALPHA_DECIMATED_FS))]
                for method, values, tau, effective_fs in [
                    ("original_500hz_tau10", filtered, 10, sfreq),
                    ("original_500hz_tau12", filtered, 12, sfreq),
                    ("decimated_50hz_tau1", decimated, 1, ALPHA_DECIMATED_FS),
                ]:
                    record = {
                        "participant_id": row.participant_id, "diagnosis": row.diagnosis,
                        "epoch_id": epoch_id, "method": method, "tau": tau,
                        "effective_sampling_hz": effective_fs,
                        "lag_ms": 1000.0 * tau / effective_fs,
                    }
                    record.update(ordinal_coordinates(values, tau))
                    resampling_records.append(record)
    if epoch_count != N_FIXED_EPOCHS:
        raise ValueError(f"{row.participant_id}: expected {N_FIXED_EPOCHS} epochs, got {epoch_count}")
    return delay_records, resampling_records

raw_start = time.time()
audits = Parallel(n_jobs=PARALLEL_JOBS, prefer="threads", verbose=5)(
    delayed(audit_subject)(row) for row in participants.itertuples(index=False)
)
epoch_delay = pd.DataFrame([record for delay_rows, _ in audits for record in delay_rows])
epoch_resampling = pd.DataFrame([record for _, resampling_rows in audits for record in resampling_rows])
assert len(epoch_delay) == 116_160
assert len(epoch_resampling) == 15_840
epoch_delay.to_csv(OUT / "epoch_ordinal_delay_audit.csv", index=False)
epoch_resampling.to_csv(OUT / "epoch_alpha_resampling_audit.csv", index=False)
print(f"Raw delay audit completed in {(time.time()-raw_start)/60:.1f} min; delay rows={len(epoch_delay):,}; resampling rows={len(epoch_resampling):,}")

### 3. Quantify sampling redundancy and the empirical autocorrelation scale

In [ ]:
subject_delay = (
    epoch_delay.groupby(["participant_id", "diagnosis", "band", "tau"], as_index=False)
    .agg(
        H=("H", "mean"), C=("C", "mean"), FIM=("FIM", "mean"),
        occupancy_fraction=("occupancy_fraction", "mean"),
        monotonic_pattern_share=("monotonic_pattern_share", "mean"),
        maximum_pattern_share=("maximum_pattern_share", "mean"),
        acf_first_nonpositive_tau=("acf_first_nonpositive_tau", "median"),
    )
)
subject_delay.to_csv(OUT / "subject_ordinal_delay_features.csv", index=False)

acf_unique = epoch_delay[["participant_id", "diagnosis", "epoch_id", "band", "acf_first_nonpositive_tau"]].drop_duplicates()
acf_summary = (
    acf_unique.groupby("band", as_index=False)
    .agg(epochs=("epoch_id", "size"), median_tau=("acf_first_nonpositive_tau", "median"),
         p25_tau=("acf_first_nonpositive_tau", lambda x: np.nanpercentile(x, 25)),
         p75_tau=("acf_first_nonpositive_tau", lambda x: np.nanpercentile(x, 75)))
)
acf_summary["prespecified_band_tau"] = acf_summary["band"].map(BAND_TAU_500)
acf_summary.to_csv(OUT / "acf_first_nonpositive_summary.csv", index=False)
display(acf_summary.style.format(precision=2))

fig, ax = plt.subplots(figsize=(9.0, 4.8))
order = list(BANDS)
sns.boxplot(data=acf_unique, x="band", y="acf_first_nonpositive_tau", order=order, showfliers=False, color="#6F92A8", ax=ax)
for x, band_name in enumerate(order):
    if band_name in BAND_TAU_500:
        ax.scatter(x, BAND_TAU_500[band_name], marker="D", s=55, color="#D18B32", edgecolor="#333333", zorder=5)
ax.set(title="Label-free autocorrelation scale by frequency band", xlabel="Band", ylabel="First non-positive ACF lag (samples at 500 Hz)")
ax.text(0.99, 0.97, "Diamonds: prespecified quarter-cycle delay", transform=ax.transAxes, ha="right", va="top", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES / "acf_zero_crossing_by_band.png", dpi=220, bbox_inches="tight")
plt.show()

### 4. Trace H, occupancy, and monotonic-pattern concentration across the delay grid

In [ ]:
detailed = subject_delay.loc[subject_delay["band"].isin(["alpha", "broadband"]) & subject_delay["tau"].isin(TAU_GRID)].copy()
delay_summary = (
    detailed.groupby(["band", "tau"], as_index=False)
    .agg(
        H_median=("H", "median"), H_p25=("H", lambda x: np.percentile(x, 25)), H_p75=("H", lambda x: np.percentile(x, 75)),
        occupancy_median=("occupancy_fraction", "median"),
        monotonic_share_median=("monotonic_pattern_share", "median"),
        FIM_median=("FIM", "median"),
    )
)
delay_summary.to_csv(OUT / "alpha_broadband_delay_grid_summary.csv", index=False)
display(delay_summary.style.format(precision=3))

fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.4))
for band_name, color in [("alpha", "#315B7D"), ("broadband", "#D18B32")]:
    plot = delay_summary.loc[delay_summary["band"] == band_name]
    axes[0].plot(plot["tau"], plot["H_median"], marker="o", color=color, label=band_name)
    axes[0].fill_between(plot["tau"], plot["H_p25"], plot["H_p75"], color=color, alpha=0.15)
    axes[1].plot(plot["tau"], plot["monotonic_share_median"], marker="o", color=color, label=band_name)
    axes[2].plot(plot["tau"], plot["occupancy_median"], marker="o", color=color, label=band_name)
axes[0].axvline(BAND_TAU_500["alpha"], color="#8A5A9E", linestyle="--", linewidth=1.3, label="alpha quarter-cycle τ=12")
axes[0].set(title="Normalized entropy", ylabel="Median H")
axes[1].set(title="Monotonic-pattern concentration", ylabel="Median ascending + descending share")
axes[2].set(title="Observed state occupancy", ylabel="Median occupied states / 24")
for ax in axes:
    ax.set(xlabel="Ordinal delay τ (samples at 500 Hz)", xticks=TAU_GRID)
    ax.legend(frameon=False, fontsize=8)
fig.suptitle("Delay dependence of temporal ordinal geometry in balanced first-60-epoch data")
fig.tight_layout()
fig.savefig(FIGURES / "alpha_broadband_delay_grid.png", dpi=220, bbox_inches="tight")
plt.show()

### 5. Compare tau=1 with a fixed band-adapted delay

Broadband is omitted from this paired comparison because it does not have a single interpretable center frequency.

In [ ]:
band_pair_rows = []
for band_name, tau_adapted in BAND_TAU_500.items():
    for tau, scheme in [(1, "tau=1"), (tau_adapted, "band-adapted")]:
        frame = subject_delay.loc[(subject_delay["band"] == band_name) & (subject_delay["tau"] == tau)]
        for row in frame.itertuples(index=False):
            band_pair_rows.append({
                "participant_id": row.participant_id, "diagnosis": row.diagnosis, "band": band_name,
                "scheme": scheme, "tau": tau, "H": row.H, "C": row.C, "FIM": row.FIM,
                "occupancy_fraction": row.occupancy_fraction, "monotonic_pattern_share": row.monotonic_pattern_share,
            })
band_pairs = pd.DataFrame(band_pair_rows)
band_adapted_summary = (
    band_pairs.groupby(["band", "scheme", "tau"], as_index=False)
    .agg(H_median=("H", "median"), C_median=("C", "median"), FIM_median=("FIM", "median"),
         occupancy_median=("occupancy_fraction", "median"), monotonic_share_median=("monotonic_pattern_share", "median"))
)
band_adapted_summary.to_csv(OUT / "band_adapted_delay_summary.csv", index=False)
display(band_adapted_summary.style.format(precision=3))

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.8))
sns.boxplot(data=band_pairs, x="band", y="H", hue="scheme", order=list(BAND_TAU_500), palette=["#315B7D", "#D18B32"], showfliers=False, ax=axes[0])
sns.boxplot(data=band_pairs, x="band", y="monotonic_pattern_share", hue="scheme", order=list(BAND_TAU_500), palette=["#315B7D", "#D18B32"], showfliers=False, ax=axes[1])
axes[0].set(title="Normalized entropy", xlabel="Band", ylabel="Participant mean H")
axes[1].set(title="Monotonic-pattern share", xlabel="Band", ylabel="Participant mean share")
for ax in axes:
    ax.legend(frameon=False, title="Delay rule", fontsize=8)
fig.suptitle("Band-adapted delay reduces sampling-redundancy signatures")
fig.tight_layout()
fig.savefig(FIGURES / "tau1_vs_band_adapted.png", dpi=220, bbox_inches="tight")
plt.show()

### 6. Test the alpha 50 Hz equivalence control

In [ ]:
subject_resampling = (
    epoch_resampling.groupby(["participant_id", "diagnosis", "method"], as_index=False)
    .agg(H=("H", "mean"), C=("C", "mean"), FIM=("FIM", "mean"),
         occupancy_fraction=("occupancy_fraction", "mean"), monotonic_pattern_share=("monotonic_pattern_share", "mean"))
)
subject_resampling.to_csv(OUT / "subject_alpha_resampling_features.csv", index=False)
wide = subject_resampling.pivot(index="participant_id", columns="method", values=["H", "C", "FIM"])
equivalence_rows = []
for coordinate in ["H", "C", "FIM"]:
    decimated = wide[(coordinate, "decimated_50hz_tau1")]
    for comparator in ["original_500hz_tau10", "original_500hz_tau12"]:
        original = wide[(coordinate, comparator)]
        equivalence_rows.append({
            "coordinate": coordinate, "comparator": comparator,
            "spearman_rho": stats.spearmanr(original, decimated).statistic,
            "mean_absolute_difference": np.mean(np.abs(original - decimated)),
            "median_original": np.median(original), "median_decimated": np.median(decimated),
        })
equivalence = pd.DataFrame(equivalence_rows)
equivalence.to_csv(OUT / "alpha_resampling_equivalence.csv", index=False)
display(equivalence.style.format(precision=4))

fig, axes = plt.subplots(1, 3, figsize=(12.0, 4.0))
for ax, coordinate in zip(axes, ["H", "C", "FIM"]):
    x = wide[(coordinate, "original_500hz_tau10")]
    y = wide[(coordinate, "decimated_50hz_tau1")]
    ax.scatter(x, y, s=28, alpha=0.75, color="#315B7D", edgecolor="white", linewidth=0.4)
    limits = [min(x.min(), y.min()), max(x.max(), y.max())]
    ax.plot(limits, limits, linestyle="--", color="#555555", linewidth=1)
    rho = stats.spearmanr(x, y).statistic
    ax.set(title=f"{coordinate}: Spearman ρ={rho:.3f}", xlabel="500 Hz, τ=10", ylabel="50 Hz, τ=1")
fig.suptitle("Alpha control: equivalent 20 ms delay before and after decimation")
fig.tight_layout()
fig.savefig(FIGURES / "alpha_resampling_equivalence.png", dpi=220, bbox_inches="tight")
plt.show()

### 7. Repeat coordinate tests with delay included in the FDR family

Each comparison contains 24 tests: eight delays times H, C, and FIM. AD uses temporal alpha and FTD uses temporal broadband. These are focused post hoc sensitivity families.

In [ ]:
def bh_adjust(p_values):
    p_values = np.asarray(p_values, float)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    output = np.empty_like(adjusted)
    output[order] = np.clip(adjusted, 0, 1)
    return output

COMPARISONS = {
    "control_vs_alzheimer": ("alzheimer", "alpha"),
    "control_vs_frontotemporal_dementia": ("frontotemporal_dementia", "broadband"),
}
test_rows = []
for comparison, (disease, band_name) in COMPARISONS.items():
    frame = subject_delay.loc[(subject_delay["band"] == band_name) & subject_delay["diagnosis"].isin(["control", disease])]
    for tau in TAU_GRID:
        subset = frame.loc[frame["tau"] == tau]
        for coordinate in ["H", "C", "FIM"]:
            control = subset.loc[subset["diagnosis"] == "control", coordinate].to_numpy()
            patient = subset.loc[subset["diagnosis"] == disease, coordinate].to_numpy()
            test = stats.mannwhitneyu(control, patient, alternative="two-sided")
            test_rows.append({
                "comparison": comparison, "band": band_name, "tau": tau, "coordinate": coordinate,
                "control_median": np.median(control), "patient_median": np.median(patient),
                "median_difference_patient_minus_control": np.median(patient) - np.median(control),
                "u_statistic": test.statistic, "p_value": test.pvalue,
            })
coordinate_tests = pd.DataFrame(test_rows)
coordinate_tests["q_value"] = coordinate_tests.groupby("comparison")["p_value"].transform(lambda x: bh_adjust(x.to_numpy()))
coordinate_tests["fdr_0_05"] = coordinate_tests["q_value"] < 0.05
coordinate_tests.to_csv(OUT / "ordinal_delay_coordinate_fdr.csv", index=False)
fdr_summary = coordinate_tests.groupby("comparison", as_index=False).agg(tests=("p_value", "size"), significant=("fdr_0_05", "sum"), minimum_q=("q_value", "min"))
fdr_summary.to_csv(OUT / "ordinal_delay_fdr_summary.csv", index=False)
display(fdr_summary.style.format(precision=4))
display(coordinate_tests.loc[coordinate_tests["fdr_0_05"]].sort_values("q_value").style.format(precision=4))

### 8. Evaluate fixed and inner-selected delays at participant level

The classifier is deliberately fixed (100-tree Extra Trees, maximum depth 3) so that the inner loop selects only the ordinal delay. Every method uses the same repeated outer partitions.

In [ ]:
def make_model(seed):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", ExtraTreesClassifier(n_estimators=N_TREES, max_depth=MODEL_DEPTH, class_weight="balanced", random_state=seed, n_jobs=1)),
    ])

def task_matrices(disease, band_name):
    base = subject_delay.loc[subject_delay["diagnosis"].isin(["control", disease]) & (subject_delay["band"] == band_name)]
    participants_order = sorted(base["participant_id"].unique())
    metadata = base[["participant_id", "diagnosis"]].drop_duplicates().set_index("participant_id").loc[participants_order].reset_index()
    y = (metadata["diagnosis"] == disease).astype(int).to_numpy()
    matrices = {}
    for tau in TAU_GRID:
        frame = base.loc[base["tau"] == tau].set_index("participant_id").loc[participants_order]
        matrices[tau] = frame[["H", "C", "FIM"]].to_numpy(float)
    return metadata, y, matrices

def run_task(comparison, disease, band_name):
    metadata, y, matrices = task_matrices(disease, band_name)
    repeat_rows, fold_rows, oof_rows = [], [], []
    for repeat in range(N_REPEATS):
        outer = StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=SEED + repeat * 1000)
        outer_splits = list(outer.split(np.zeros(len(y)), y))
        for tau in TAU_GRID:
            probabilities = np.zeros(len(y), float)
            for fold, (train_idx, test_idx) in enumerate(outer_splits, start=1):
                model = make_model(SEED + repeat * 1000 + fold + tau)
                model.fit(matrices[tau][train_idx], y[train_idx])
                probabilities[test_idx] = model.predict_proba(matrices[tau][test_idx])[:, 1]
                fold_rows.append({"comparison": comparison, "method": f"fixed_tau{tau}", "repeat": repeat, "fold": fold, "selected_tau": tau})
            prediction = (probabilities >= 0.5).astype(int)
            repeat_rows.append({"comparison": comparison, "method": f"fixed_tau{tau}", "repeat": repeat,
                                "roc_auc": roc_auc_score(y, probabilities), "balanced_accuracy": balanced_accuracy_score(y, prediction)})
            for index, row in metadata.iterrows():
                oof_rows.append({"participant_id": row.participant_id, "diagnosis": row.diagnosis, "comparison": comparison,
                                 "method": f"fixed_tau{tau}", "repeat": repeat, "y_true": int(y[index]),
                                 "probability_positive": probabilities[index], "y_pred": int(prediction[index])})

        probabilities = np.zeros(len(y), float)
        for fold, (train_idx, test_idx) in enumerate(outer_splits, start=1):
            inner = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=SEED + repeat * 1000 + fold + 500)
            scores = []
            for tau in TAU_GRID:
                inner_scores = []
                for inner_fold, (inner_train, inner_test) in enumerate(inner.split(matrices[tau][train_idx], y[train_idx]), start=1):
                    model = make_model(SEED + repeat * 1000 + fold * 10 + inner_fold + tau)
                    model.fit(matrices[tau][train_idx][inner_train], y[train_idx][inner_train])
                    probability = model.predict_proba(matrices[tau][train_idx][inner_test])[:, 1]
                    inner_scores.append(roc_auc_score(y[train_idx][inner_test], probability))
                scores.append((float(np.mean(inner_scores)), tau))
            selected_score, selected_tau = sorted(scores, key=lambda item: (-item[0], item[1]))[0]
            model = make_model(SEED + repeat * 1000 + fold + selected_tau)
            model.fit(matrices[selected_tau][train_idx], y[train_idx])
            probabilities[test_idx] = model.predict_proba(matrices[selected_tau][test_idx])[:, 1]
            fold_rows.append({"comparison": comparison, "method": "inner_selected_tau", "repeat": repeat, "fold": fold,
                              "selected_tau": selected_tau, "inner_auc": selected_score})
        prediction = (probabilities >= 0.5).astype(int)
        repeat_rows.append({"comparison": comparison, "method": "inner_selected_tau", "repeat": repeat,
                            "roc_auc": roc_auc_score(y, probabilities), "balanced_accuracy": balanced_accuracy_score(y, prediction)})
        for index, row in metadata.iterrows():
            oof_rows.append({"participant_id": row.participant_id, "diagnosis": row.diagnosis, "comparison": comparison,
                             "method": "inner_selected_tau", "repeat": repeat, "y_true": int(y[index]),
                             "probability_positive": probabilities[index], "y_pred": int(prediction[index])})
    return repeat_rows, fold_rows, oof_rows

classification_start = time.time()
outputs = Parallel(n_jobs=2, prefer="threads", verbose=5)(
    delayed(run_task)(comparison, disease, band_name) for comparison, (disease, band_name) in COMPARISONS.items()
)
classification_metrics = pd.DataFrame([row for output in outputs for row in output[0]])
classification_folds = pd.DataFrame([row for output in outputs for row in output[1]])
classification_oof = pd.DataFrame([row for output in outputs for row in output[2]])
classification_metrics.to_csv(OUT / "repeated_delay_classification_metrics.csv", index=False)
classification_folds.to_csv(OUT / "repeated_delay_classification_folds.csv", index=False)
classification_oof.to_csv(OUT / "repeated_delay_classification_oof.csv", index=False)
print(f"Classification completed in {(time.time()-classification_start)/60:.1f} min")

### 9. Summarize predictive sensitivity and fold-local delay choices

In [ ]:
classification_summary = (
    classification_metrics.groupby(["comparison", "method"], as_index=False)
    .agg(repetitions=("repeat", "size"), auc_mean=("roc_auc", "mean"), auc_median=("roc_auc", "median"),
         auc_sd=("roc_auc", "std"), auc_p2_5=("roc_auc", lambda x: np.percentile(x, 2.5)),
         auc_p97_5=("roc_auc", lambda x: np.percentile(x, 97.5)),
         balanced_accuracy_mean=("balanced_accuracy", "mean"))
)
classification_summary.to_csv(OUT / "repeated_delay_classification_summary.csv", index=False)
display(classification_summary.style.format(precision=3))

selection_frequency = (
    classification_folds.loc[classification_folds["method"] == "inner_selected_tau"]
    .groupby(["comparison", "selected_tau"], as_index=False).size().rename(columns={"size": "selected_outer_folds"})
)
selection_frequency["selection_fraction"] = selection_frequency.groupby("comparison")["selected_outer_folds"].transform(lambda x: x / x.sum())
selection_frequency.to_csv(OUT / "inner_fold_tau_selection_frequency.csv", index=False)
display(selection_frequency.style.format({"selection_fraction": "{:.1%}"}))

fig, axes = plt.subplots(1, 2, figsize=(13.2, 4.8), sharey=True)
method_order = [f"fixed_tau{tau}" for tau in TAU_GRID] + ["inner_selected_tau"]
palette = [COLORS[tau] for tau in TAU_GRID] + [COLORS["adaptive"]]
for ax, comparison in zip(axes, COMPARISONS):
    plot = classification_metrics.loc[classification_metrics["comparison"] == comparison]
    sns.boxplot(data=plot, x="method", y="roc_auc", order=method_order, hue="method", palette=dict(zip(method_order, palette)), legend=False, showfliers=False, ax=ax)
    sns.stripplot(data=plot, x="method", y="roc_auc", order=method_order, color="#222222", size=3.2, jitter=0.10, ax=ax)
    ax.axhline(0.5, color="#666666", linestyle="--", linewidth=1)
    ax.set(title=comparison.replace("_", " "), xlabel="Delay strategy", ylabel="Out-of-fold ROC AUC")
    ax.tick_params(axis="x", rotation=55)
fig.suptitle("Repeated participant-level sensitivity to ordinal delay")
fig.tight_layout()
fig.savefig(FIGURES / "repeated_delay_auc.png", dpi=220, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9.0, 4.8))
sns.barplot(data=selection_frequency, x="comparison", y="selection_fraction", hue="selected_tau",
            palette={tau: COLORS[tau] for tau in TAU_GRID}, ax=ax)
ax.set(title="Delay selected exclusively inside outer training partitions", xlabel="Comparison", ylabel="Fraction of outer folds", ylim=(0, 1))
ax.tick_params(axis="x", rotation=15)
ax.legend(title="Selected τ", frameon=False, ncol=4)
fig.tight_layout()
fig.savefig(FIGURES / "inner_fold_tau_selection_frequency.png", dpi=220, bbox_inches="tight")
plt.show()

## Takeaways

In [ ]:
display(Markdown(
    "1. **The tau=1 alpha geometry is explicitly tested as a sampling-scale representation, not assumed to be invariant physiology.**\n"
    "2. **Band-adapted delays are fixed from sampling rate and band definitions, without diagnostic labels.**\n"
    "3. **The 50 Hz control tests whether an equivalent physical lag reproduces the 500 Hz geometry.**\n"
    "4. **Any predictive choice of tau is estimated only inside training folds; fixed-tau comparisons remain sensitivity estimates.**\n"
    "5. **Because temporal alpha and broadband were highlighted by the primary analysis, their inferential and predictive results remain post hoc and require external validation.**"
))

### Save execution manifest

In [ ]:
manifest = {
    "project_root": str(PROJECT_ROOT), "dataset_root": str(DATASET_ROOT), "output": str(OUT),
    "seed": SEED, "fixed_epochs": N_FIXED_EPOCHS, "m": M, "tau_grid": list(TAU_GRID),
    "band_tau_500hz": BAND_TAU_500, "alpha_decimated_sampling_hz": ALPHA_DECIMATED_FS,
    "n_repeats": N_REPEATS, "outer_folds": OUTER_FOLDS, "inner_folds": INNER_FOLDS,
    "classifier": {"name": "ExtraTreesClassifier", "n_trees": N_TREES, "max_depth": MODEL_DEPTH},
    "python": sys.version,
    "package_versions": {"numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__},
    "outputs": sorted(str(path.relative_to(OUT)) for path in OUT.rglob("*") if path.is_file()),
}
(OUT / "notebook_09_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
display(manifest)